# 07 — SARIMAX com variáveis externas

Nos notebooks anteriores, treinamos ARIMA e SARIMA usando apenas a própria série de demanda.

Agora vamos testar uma evolução: o SARIMAX.

O SARIMAX permite adicionar variáveis externas ao modelo, como:

- temperatura;
- umidade;
- vento;
- feriado;
- dia útil;
- dia da semana.

A pergunta agora é:

> será que o modelo melhora quando usamos informações do contexto?

### 07.01 Imports e caminhos

In [38]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import plotly.graph_objects as go

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import (
    CORES,
    aplicar_layout_padrao,
    grafico_linha_padrao
)

caminho_outputs_tabelas = raiz_projeto / "outputs" / "tabelas"

caminho_base_organizada = caminho_outputs_tabelas / "base_bike_organizada.csv"
caminho_comparacao_modelos = caminho_outputs_tabelas / "comparacao_modelos.csv"
caminho_previsoes_arima = caminho_outputs_tabelas / "previsoes_arima.csv"
caminho_previsoes_sarima = caminho_outputs_tabelas / "previsoes_sarima.csv"

print("Base organizada:")
print(caminho_base_organizada)

Base organizada:
c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\base_bike_organizada.csv


### 07.02 Carregar base organizada

In [39]:
df = pd.read_csv(caminho_base_organizada)

df["data_hora"] = pd.to_datetime(df["data_hora"])

df = df.sort_values("data_hora").reset_index(drop=True)

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

df.head()

Linhas: 10886
Colunas: 18


,data_hora,estacao,feriado,dia_util,clima,temperatura,sensacao_termica,umidade,velocidade_vento,usuarios_casuais,usuarios_registrados,demanda,ano,mes,dia,hora,dia_semana,nome_dia_semana
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,2011,1,1,0,5,sábado
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,2011,1,1,1,5,sábado
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,2011,1,1,2,5,sábado
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,2011,1,1,3,5,sábado
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,2011,1,1,4,5,sábado


## Por que usar variáveis externas?

Até agora, ARIMA e SARIMA usaram apenas o histórico da demanda.

Mas, no mundo real, a demanda de bicicletas pode depender também de fatores externos.

Por exemplo:

- em dias muito frios, a demanda pode cair;
- em dias úteis, o padrão pode ser diferente;
- em feriados, o comportamento pode mudar;
- clima e umidade também podem influenciar o uso.

### 07.03 Criar base diária com variáveis externas

In [40]:
base_diaria = (
    df.assign(data=df["data_hora"].dt.floor("D"))
    .groupby("data", as_index=False)
    .agg(
        demanda=("demanda", "sum"),
        temperatura_media=("temperatura", "mean"),
        sensacao_termica_media=("sensacao_termica", "mean"),
        umidade_media=("umidade", "mean"),
        velocidade_vento_media=("velocidade_vento", "mean"),
        feriado=("feriado", "max"),
        dia_util=("dia_util", "max"),
        dia_semana=("dia_semana", "first"),
        mes=("mes", "first")
    )
    .rename(columns={"data": "data_hora"})
)

print(f"Quantidade de dias observados: {base_diaria.shape[0]}")

base_diaria.head()

Quantidade de dias observados: 456


,data_hora,demanda,temperatura_media,sensacao_termica_media,umidade_media,velocidade_vento_media,feriado,dia_util,dia_semana,mes
0,2011-01-01,985,14.110833,18.181250,80.583333,10.749871,0,0,5,1
1,2011-01-02,801,14.902609,17.686957,69.608696,16.652122,0,0,6,1
2,2011-01-03,1349,8.050909,9.470227,43.727273,16.636709,0,1,0,1
3,2011-01-04,1562,8.200000,10.606087,59.043478,10.739809,0,1,1,1
4,2011-01-05,1600,9.305217,11.463478,43.695652,12.522300,0,1,2,1


### 07.04 Criar variáveis de dia da semana

In [41]:
dummies_dia_semana = pd.get_dummies(
    base_diaria["dia_semana"],
    prefix="dia_semana",
    drop_first=True
)

base_modelagem = pd.concat(
    [
        base_diaria,
        dummies_dia_semana
    ],
    axis=1
)

base_modelagem.head()

,data_hora,demanda,temperatura_media,sensacao_termica_media,umidade_media,velocidade_vento_media,feriado,dia_util,dia_semana,mes,dia_semana_1,dia_semana_2,dia_semana_3,dia_semana_4,dia_semana_5,dia_semana_6
0,2011-01-01,985,14.110833,18.181250,80.583333,10.749871,0,0,5,1,False,False,False,False,True,False
1,2011-01-02,801,14.902609,17.686957,69.608696,16.652122,0,0,6,1,False,False,False,False,False,True
2,2011-01-03,1349,8.050909,9.470227,43.727273,16.636709,0,1,0,1,False,False,False,False,False,False
3,2011-01-04,1562,8.200000,10.606087,59.043478,10.739809,0,1,1,1,True,False,False,False,False,False
4,2011-01-05,1600,9.305217,11.463478,43.695652,12.522300,0,1,2,1,False,True,False,False,False,False


### 07.05 Definir variáveis externas

In [42]:
variaveis_externas = [
    "temperatura_media",
    "sensacao_termica_media",
    "umidade_media",
    "velocidade_vento_media",
    "feriado",
    "dia_util"
] + list(dummies_dia_semana.columns)

print("Variáveis externas usadas no SARIMAX:")
variaveis_externas

Variáveis externas usadas no SARIMAX:


['temperatura_media',
 'sensacao_termica_media',
 'umidade_media',
 'velocidade_vento_media',
 'feriado',
 'dia_util',
 'dia_semana_1',
 'dia_semana_2',
 'dia_semana_3',
 'dia_semana_4',
 'dia_semana_5',
 'dia_semana_6']

### 07.06 Separar treino e teste

In [43]:
tamanho_teste = 60

treino = base_modelagem.iloc[:-tamanho_teste].copy()
teste = base_modelagem.iloc[-tamanho_teste:].copy()

y_treino = treino["demanda"].astype(float)
y_teste = teste["demanda"].astype(float)

X_treino = treino[variaveis_externas].astype(float)
X_teste = teste[variaveis_externas].astype(float)

print("Tamanho do treino:", treino.shape[0])
print("Tamanho do teste:", teste.shape[0])

print("\nPeríodo de treino:")
print(treino["data_hora"].min(), "até", treino["data_hora"].max())

print("\nPeríodo de teste:")
print(teste["data_hora"].min(), "até", teste["data_hora"].max())

Tamanho do treino: 396
Tamanho do teste: 60

Período de treino:
2011-01-01 00:00:00 até 2012-09-16 00:00:00

Período de teste:
2012-09-17 00:00:00 até 2012-12-19 00:00:00


## Treinando o SARIMAX

O SARIMAX mantém a lógica do SARIMA, mas adiciona variáveis externas.

Vamos começar com a mesma estrutura sazonal do notebook anterior:

- `order = (1, 1, 1)`
- `seasonal_order = (1, 0, 1, 7)`

A diferença agora é que o modelo também recebe `X_treino`.

### 07.07 Treinar SARIMAX

In [44]:
order = (1, 1, 1)
seasonal_order = (1, 0, 1, 7)

modelo_sarimax = SARIMAX(
    y_treino,
    exog=X_treino,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

resultado_sarimax = modelo_sarimax.fit(disp=False)

resultado_sarimax.summary()

c:\GitHub\ALURA-TESTE\series-temporais-bikes\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                     
=========================================================================================
Dep. Variable:                           demanda   No. Observations:                  396
Model:             SARIMAX(1, 1, 1)x(1, 0, 1, 7)   Log Likelihood               -3071.576
Date:                           Fri, 26 Jun 2026   AIC                           6177.153
Time:                                   12:10:13   BIC                           6244.402
Sample:                                        0   HQIC                          6203.821
                                           - 396                                         
Covariance Type:                             opg                                         
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
temperatura_media         64.7589     38.835      1.668      0.095     -11.356     140.874
sensacao_termica_media    21.2825     33.781      0.630      0.529     -44.927      87.492
umidade_media            -26.1733      2.351    -11.132      0.000     -30.782     -21.565
velocidade_vento_media   -45.3092      8.785     -5.158      0.000     -62.527     -28.091
feriado                   -9.0715    195.627     -0.046      0.963    -392.494     374.351
dia_util                 -63.7001    111.118     -0.573      0.566    -281.488     154.088
dia_semana_1             159.2887    169.339      0.941      0.347    -172.609     491.187
dia_semana_2              33.0630    142.207      0.232      0.816    -245.658     311.784
dia_semana_3             103.4605    159.162      0.650      0.516    -208.492     415.413
dia_semana_4             182.3902    163.549      1.115      0.265    -138.160     502.940
dia_semana_5             217.0442    102.953      2.108      0.035      15.260     418.829
dia_semana_6            -144.2725     94.174     -1.532      0.126    -328.850      40.305
ar.L1                      0.2548      0.067      3.783      0.000       0.123       0.387
ma.L1                     -0.8564      0.035    -24.536      0.000      -0.925      -0.788
ar.S.L7                   -0.0201      4.927     -0.004      0.997      -9.676       9.636
ma.S.L7                    0.0274      4.924      0.006      0.996      -9.624       9.679
sigma2                  5.205e+05   3.07e+04     16.966      0.000     4.6e+05    5.81e+05
===================================================================================
Ljung-Box (L1) (Q):                   0.10   Jarque-Bera (JB):               182.79
Prob(Q):                              0.76   Prob(JB):                         0.00
Heteroskedasticity (H):               2.09   Skew:                            -0.70
Prob(H) (two-sided):                  0.00   Kurtosis:                         6.06
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
[2] Covariance matrix is singular or near-singular, with condition number 1.75e+21. Standard errors may be unstable.
"""

### 07.08 Gerar previsões no teste

In [45]:
previsao_sarimax = resultado_sarimax.forecast(
    steps=len(teste),
    exog=X_teste
)

teste_resultado = teste[["data_hora", "demanda"]].copy()
teste_resultado["previsao_sarimax"] = previsao_sarimax.values

teste_resultado.head()

,data_hora,demanda,previsao_sarimax
396,2012-09-17,6869,6900.850096
397,2012-09-18,4073,6227.377995
398,2012-09-19,7591,7208.963856
399,2012-10-01,6778,7145.538681
400,2012-10-02,4639,6900.933922


### 07.09 Visualizar real versus SARIMAX

In [46]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=teste_resultado["data_hora"],
        y=teste_resultado["demanda"],
        mode="lines",
        name="Real",
        line=dict(color=CORES["azul_escuro"], width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=teste_resultado["data_hora"],
        y=teste_resultado["previsao_sarimax"],
        mode="lines",
        name="SARIMAX",
        line=dict(color=CORES["azul_principal"], width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Real versus previsto — SARIMAX",
    altura=500
)

fig.show()

### 07.10 Função de métricas

In [47]:
def calcular_metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))

    y_real = np.array(y_real)
    y_pred = np.array(y_pred)

    mascara = y_real != 0
    mape = np.mean(
        np.abs((y_real[mascara] - y_pred[mascara]) / y_real[mascara])
    ) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

### 07.11 Calcular métricas do SARIMAX

In [48]:
metricas_sarimax = calcular_metricas(
    teste_resultado["demanda"],
    teste_resultado["previsao_sarimax"]
)

df_metricas_sarimax = pd.DataFrame([
    {
        "modelo": "SARIMAX(1,1,1)(1,0,1,7)",
        **metricas_sarimax
    }
])

df_metricas_sarimax

,modelo,MAE,RMSE,MAPE
0,"SARIMAX(1,1,1)(1,0,1,7)",908.079882,1092.154192,18.141702


### 07.12 Carregar comparação anterior

In [49]:
comparacao_modelos = pd.read_csv(caminho_comparacao_modelos)

comparacao_modelos

,modelo,MAE,RMSE,MAPE
0,Baseline,1585.183333,1865.908943,31.813723
1,"ARIMA(1,1,1)",1730.649703,2031.357449,34.679064
2,"SARIMA(1,1,1)(1,0,1,7)",1775.685477,2077.504788,35.530606


### 07.13 Comparar todos os modelos

In [50]:
comparacao_final = pd.concat(
    [
        comparacao_modelos,
        df_metricas_sarimax
    ],
    ignore_index=True
)

comparacao_final = (
    comparacao_final
    .sort_values("MAPE")
    .reset_index(drop=True)
)

comparacao_final

,modelo,MAE,RMSE,MAPE
0,"SARIMAX(1,1,1)(1,0,1,7)",908.079882,1092.154192,18.141702
1,Baseline,1585.183333,1865.908943,31.813723
2,"ARIMA(1,1,1)",1730.649703,2031.357449,34.679064
3,"SARIMA(1,1,1)(1,0,1,7)",1775.685477,2077.504788,35.530606


### 07.14 Visualizar comparação final

In [51]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=comparacao_final["modelo"],
        y=comparacao_final["MAPE"],
        marker_color=CORES["azul_principal"],
        name="MAPE"
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Comparação final dos modelos pelo MAPE",
    altura=450,
    legenda_horizontal=False
)

fig.update_xaxes(title="Modelo")
fig.update_yaxes(title="MAPE (%)")

fig.show()

#### 07.15 Melhor modelo

In [52]:
melhor_modelo = comparacao_final.iloc[0]

nome_melhor_modelo = melhor_modelo["modelo"]

print("Melhor modelo na comparação final:")
print(nome_melhor_modelo)

print("\nMAPE:")
print(f"{melhor_modelo['MAPE']:.2f}%")

Melhor modelo na comparação final:
SARIMAX(1,1,1)(1,0,1,7)

MAPE:
18.14%


## Atenção sobre previsão futura

O SARIMAX usa variáveis externas.

Isso melhora o modelo, mas traz uma responsabilidade:

> para prever o futuro, precisamos ter também valores futuros das variáveis externas.

Por exemplo, se usamos temperatura e umidade, precisamos ter uma previsão ou um cenário para essas variáveis.

Neste projeto, vamos salvar a comparação final e as previsões do teste. A previsão futura poderá ser apresentada como um cenário simples na etapa final.

### 07.16 Consolidar previsões do teste

In [53]:
previsoes_arima = pd.read_csv(caminho_previsoes_arima)
previsoes_sarima = pd.read_csv(caminho_previsoes_sarima)

previsoes_arima["data_hora"] = pd.to_datetime(previsoes_arima["data_hora"])
previsoes_sarima["data_hora"] = pd.to_datetime(previsoes_sarima["data_hora"])

previsoes_teste = previsoes_arima[
    [
        "data_hora",
        "demanda",
        "previsao_baseline",
        "previsao_arima"
    ]
].copy()

previsoes_teste = previsoes_teste.merge(
    previsoes_sarima[["data_hora", "previsao_sarima"]],
    on="data_hora",
    how="left"
)

previsoes_teste = previsoes_teste.merge(
    teste_resultado[["data_hora", "previsao_sarimax"]],
    on="data_hora",
    how="left"
)

previsoes_teste = previsoes_teste.rename(
    columns={"demanda": "demanda_real"}
)

previsoes_teste.head()

,data_hora,demanda_real,previsao_baseline,previsao_arima,previsao_sarima,previsao_sarimax
0,2012-09-17,6869,7333,7486.108513,7486.740921,6900.850096
1,2012-09-18,4073,7333,7524.392858,7571.036681,6227.377995
2,2012-09-19,7591,7333,7533.965749,7587.662987,7208.963856
3,2012-10-01,6778,7333,7536.359422,7572.316395,7145.538681
4,2012-10-02,4639,7333,7536.957954,7590.249756,6900.933922


### 07.17 Visualizar previsões no teste

In [54]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["demanda_real"],
        mode="lines",
        name="Real",
        line=dict(color="#0B1F4D", width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_baseline"],
        mode="lines",
        name="Baseline",
        line=dict(color="#7A7A7A", width=2, dash="dash")
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_arima"],
        mode="lines",
        name="ARIMA",
        line=dict(color="#6FA8FF", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_sarima"],
        mode="lines",
        name="SARIMA",
        line=dict(color="#F28E2B", width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=previsoes_teste["data_hora"],
        y=previsoes_teste["previsao_sarimax"],
        mode="lines",
        name="SARIMAX",
        line=dict(color="#2CA02C", width=3)
    )
)

fig = aplicar_layout_padrao(
    fig,
    titulo="Comparação das previsões no período de teste",
    altura=500
)

fig.show()

### 07.18 Comparação Final

In [55]:
baseline_mape = comparacao_final.loc[
    comparacao_final["modelo"] == "Baseline",
    "MAPE"
].iloc[0]

sarimax_mape = comparacao_final.loc[
    comparacao_final["modelo"].str.contains("SARIMAX"),
    "MAPE"
].iloc[0]

reducao_mape = ((baseline_mape - sarimax_mape) / baseline_mape) * 100

print(f"MAPE da baseline: {baseline_mape:.2f}%")
print(f"MAPE do SARIMAX: {sarimax_mape:.2f}%")
print(f"Redução aproximada do erro percentual: {reducao_mape:.1f}%")

MAPE da baseline: 31.81%
MAPE do SARIMAX: 18.14%
Redução aproximada do erro percentual: 43.0%


### 07.18 Salvar resultados finais

In [56]:
caminho_metricas_final = caminho_outputs_tabelas / "metricas.csv"
caminho_previsoes_teste_final = caminho_outputs_tabelas / "previsoes_teste.csv"
caminho_metricas_sarimax = caminho_outputs_tabelas / "metricas_sarimax.csv"
caminho_previsoes_sarimax = caminho_outputs_tabelas / "previsoes_sarimax.csv"
caminho_serie_historica = caminho_outputs_tabelas / "serie_historica.csv"

comparacao_final.to_csv(
    caminho_metricas_final,
    index=False,
    encoding="utf-8-sig"
)

df_metricas_sarimax.to_csv(
    caminho_metricas_sarimax,
    index=False,
    encoding="utf-8-sig"
)

teste_resultado.to_csv(
    caminho_previsoes_sarimax,
    index=False,
    encoding="utf-8-sig"
)

previsoes_teste.to_csv(
    caminho_previsoes_teste_final,
    index=False,
    encoding="utf-8-sig"
)

base_diaria[["data_hora", "demanda"]].to_csv(
    caminho_serie_historica,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos salvos:")
print("-", caminho_metricas_final)
print("-", caminho_previsoes_teste_final)
print("-", caminho_metricas_sarimax)
print("-", caminho_previsoes_sarimax)
print("-", caminho_serie_historica)

Arquivos salvos:
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\metricas.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\previsoes_teste.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\metricas_sarimax.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\previsoes_sarimax.csv
- c:\GitHub\ALURA-TESTE\series-temporais-bikes\outputs\tabelas\serie_historica.csv


## Leitura do resultado

O SARIMAX apresentou o menor erro entre os modelos testados.

Isso mostra que, para este problema, variáveis externas como temperatura, umidade, vento, feriado e dia útil ajudam o modelo a representar melhor a demanda.

A principal lição é: modelos de séries temporais não dependem apenas do algoritmo, mas também da qualidade das informações usadas.

## Próximo passo

Agora temos uma comparação mais completa.

Vimos que ARIMA e SARIMA usam apenas o passado da série, enquanto o SARIMAX permite adicionar informações externas do problema.

Na próxima etapa, vamos organizar os arquivos finais para apresentar o projeto em uma aplicação Streamlit.